# Kitchen Assistant — Live App Demo

This notebook imports and runs the **real** `app/` modules directly — no mocked internals, no copied code. It exercises:

1. `StateManager` — atomic per-session state
2. `cooking_tools` — timers, unit conversion, recipe scaling — called directly
3. `ToolRegistry` — the same name+args dispatch path the Gemini Live gateway uses
4. `TimerEngine` — a real `asyncio` countdown with proactive-expiry callback
5. `RecipeStore` — semantic search over the real `data/recipes.db` (needs `GOOGLE_API_KEY`)
6. An end-to-end scenario chaining all of the above

**Not covered here:** the Gemini Live audio session itself (`LiveGateway`) — that needs a microphone or a prerecorded WAV. See `scripts/live_smoke.py` for that path, or run `poetry run uvicorn app.main:app --reload` and talk to it in a browser.

Run cells top to bottom. Cells in section 5 need a valid `GOOGLE_API_KEY` in `.env`; everything else runs offline.

## 1. Setup

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "app").exists() else Path.cwd().parent
assert (ROOT / "app").exists(), f"Could not locate repo root from {Path.cwd()}"
sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

import json

from app.state_manager import StateManager
from app.services.timer_engine import TimerEngine
from app.services.recipe_store import RecipeStore
from app.tools.registry import ToolRegistry
from app.tools import cooking_tools

def show(label, result):
    print(f"{label} ->")
    print(json.dumps(result, indent=2, default=str))
    print()

## 2. State manager

`StateManager.update()` is the single atomic read-modify-write path every tool goes through (`app/state_manager.py`). A fresh in-memory instance here mirrors the app's default (`USE_REDIS=false`).

In [2]:
state_manager = StateManager(use_redis=False)
session_id = "demo-session-1"

state = await state_manager.get_or_create_state(session_id)
print("Fresh session state:")
print(state.model_dump_json(indent=2))

Fresh session state:
{
  "session_id": "demo-session-1",
  "recipe_id": null,
  "recipe_metadata": null,
  "current_step_index": 0,
  "active_timers": {},
  "servings_multiplier": 1.0,
  "last_updated": "2026-07-14T20:09:59.392252"
}


## 3. Cooking tools, called directly

Every tool in `app/tools/cooking_tools.py` takes its state/session dependencies as explicit arguments (no module-level globals — Phase 1 hardening). We call a few directly, the same way `tests/test_cooking_tools.py` does.

In [3]:
result = await cooking_tools.set_kitchen_timer(
    state_manager, session_id, duration_seconds=600, label="stock reduction"
)
show("set_kitchen_timer", result)

conv = await cooking_tools.convert_units(value=2, from_unit="cup", to_unit="ml")
show("convert_units(2 cup -> ml)", conv)

bad = await cooking_tools.convert_units(value=1, from_unit="lightyear", to_unit="ml")
show("convert_units(unsupported unit) — structured error, not an exception", bad)

set_kitchen_timer ->
{
  "status": "success",
  "timer_id": "eea739a3",
  "label": "stock reduction",
  "duration": 600
}

convert_units(2 cup -> ml) ->
{
  "status": "success",
  "value": 473.18,
  "unit": "ml"
}

convert_units(unsupported unit) — structured error, not an exception ->
{
  "status": "error",
  "message": "Conversion from lightyear to ml not supported."
}



## 4. Tool registry dispatch

`ToolRegistry.dispatch(session_id, name, args)` is exactly what `LiveGateway._handle_tool_call` invokes when Gemini requests a tool call. `session_id`/`state_manager`/etc. are injected by signature inspection — never model-visible parameters.

In [4]:
timer_engine = TimerEngine(state_manager)
recipe_store = RecipeStore(db_path=str(ROOT / "data" / "recipes.db"))
registry = ToolRegistry(state_manager, timer_engine=timer_engine, recipe_store=recipe_store)

print("Registered tools:", [d.name for d in registry.declarations])

dispatch_result = await registry.dispatch(
    session_id, "list_timers", {}
)
show("registry.dispatch('list_timers')", dispatch_result)

Registered tools: ['set_kitchen_timer', 'cancel_timer', 'list_timers', 'convert_units', 'scale_recipe', 'search_recipes', 'load_recipe', 'navigate_steps']
registry.dispatch('list_timers') ->
{
  "status": "success",
  "timers": [
    {
      "timer_id": "eea739a3",
      "label": "stock reduction",
      "remaining_seconds": 600
    }
  ]
}



## 5. Timer engine — a real countdown

`TimerEngine` runs actual `asyncio` countdown tasks (`app/services/timer_engine.py`). The gateway registers a per-session callback so the assistant can *speak* the expiry unprompted (Phase 4); here we register a plain notebook callback to observe the same event.

In [5]:
import asyncio

expired = asyncio.Event()

async def on_expiry(timer):
    print(f"[proactive nudge] Timer '{timer.label}' just expired — announce it now.")
    expired.set()

timer_engine.register_session(session_id, on_expiry)

quick_timer = await cooking_tools.set_kitchen_timer(
    state_manager, session_id, duration_seconds=5, label="quick demo timer", timer_engine=timer_engine
)
show("set_kitchen_timer (5s, engine-backed)", quick_timer)

print("Waiting for it to expire...")
await asyncio.wait_for(expired.wait(), timeout=10)

timer_engine.unregister_session(session_id)

set_kitchen_timer (5s, engine-backed) ->
{
  "status": "success",
  "timer_id": "4f39c060",
  "label": "quick demo timer",
  "duration": 5
}

Waiting for it to expire...


[proactive nudge] Timer 'quick demo timer' just expired — announce it now.


## 6. RAG search over the real recipe catalog

`RecipeStore` embeds the query with `gemini-embedding-001` and runs `array_distance` over `data/recipes.db` (DuckDB + vss, Phase 5). **Requires `GOOGLE_API_KEY` in `.env`** — skip this cell if you don't have one; everything above and below (except the load/navigate cell, which needs a real recipe id) still works.

In [6]:
search_result = await cooking_tools.search_recipes(recipe_store, query="something with mushrooms", k=3)
show("search_recipes('something with mushrooms')", search_result)

search_recipes('something with mushrooms') ->
{
  "status": "success",
  "results": [
    {
      "id": "r2",
      "title": "Mushroom Risotto",
      "total_time_minutes": 45
    },
    {
      "id": "r15",
      "title": "Creamy Tomato Basil Soup",
      "total_time_minutes": 25
    },
    {
      "id": "r8",
      "title": "Vegetable Pad Thai",
      "total_time_minutes": 25
    }
  ]
}



## 7. End-to-end scenario

One narrative chain — search → load → scale → navigate — mirroring the sequence of tool calls a real voice conversation would trigger, each result printed as the model would see it.

In [7]:
scenario_session = "demo-session-2"

found = await registry.dispatch(scenario_session, "search_recipes", {"query": "pasta", "k": 1})
show("1. search_recipes('pasta')", found)

recipe_id = found["results"][0]["id"]
loaded = await registry.dispatch(scenario_session, "load_recipe", {"recipe_id": recipe_id})
show("2. load_recipe", loaded)

scaled = await registry.dispatch(scenario_session, "scale_recipe", {"multiplier": 2.0})
show("3. scale_recipe(2.0)", scaled)

timer_set = await registry.dispatch(
    scenario_session, "set_kitchen_timer", {"duration_seconds": 540, "label": "pasta water"}
)
show("4. set_kitchen_timer", timer_set)

step = await registry.dispatch(scenario_session, "navigate_steps", {"direction": "next"})
show("5. navigate_steps('next')", step)

final_state = await state_manager.get_state(scenario_session)
print("Final session state:")
print(final_state.model_dump_json(indent=2))

1. search_recipes('pasta') ->
{
  "status": "success",
  "results": [
    {
      "id": "r1",
      "title": "Classic Carbonara",
      "total_time_minutes": 20
    }
  ]
}

2. load_recipe ->
{
  "status": "success",
  "recipe_id": "r1",
  "title": "Classic Carbonara",
  "total_steps": 1,
  "first_instruction": "Boil water."
}

3. scale_recipe(2.0) ->
{
  "status": "success",
  "new_multiplier": 2.0,
  "scaled_ingredients": [
    {
      "name": "Pasta",
      "amount": 400.0,
      "unit": "g"
    }
  ]
}

4. set_kitchen_timer ->
{
  "status": "success",
  "timer_id": "fe849331",
  "label": "pasta water",
  "duration": 540
}

5. navigate_steps('next') ->
{
  "status": "success",
  "current_step": 0,
  "total_steps": 1,
  "instruction": "Boil water."
}

Final session state:
{
  "session_id": "demo-session-2",
  "recipe_id": "r1",
  "recipe_metadata": {
    "id": "r1",
    "title": "Classic Carbonara",
    "description": null,
    "ingredients": [
      {
        "name": "Pasta",
      

## What's not covered

This notebook drives every tool and service the Gemini Live gateway calls into, but not the gateway's audio loop itself (`app/live/gateway.py` `LiveGateway`) — that needs either:

- a microphone + browser session: `poetry run uvicorn app.main:app --reload`, then open `http://localhost:8000`, or
- a prerecorded WAV smoke test: `poetry run python scripts/live_smoke.py`

`tests/test_gateway.py` and `tests/test_main.py` cover that audio-adjacent protocol logic (tool dispatch, barge-in, reconnect) against a fake Live backend, with no microphone or API key required.